# Kabyle AI — premier test sur Google Colab

Notebook de démarrage : vérification GPU, récupération privée du code, contrôle du tokenizer Qwen3-1.7B et validation d'un fichier JSONL de données **déjà autorisées**. Aucun entraînement ne démarre automatiquement.

**Confidentialité :** le dépôt reste privé. Dans Colab, ajoute un secret `GITHUB_TOKEN` contenant un token GitHub fine-grained limité à ce dépôt avec `Contents: Read-only`, puis active l'accès du notebook à ce secret. Ne colle jamais le token dans une cellule.

Le corpus et le fichier JSONL ne doivent contenir que des lignes dont les droits d'entraînement ont été vérifiés. Ne téléverse pas de benchmark dev/test.


In [ ]:
from google.colab import userdata
import io, os, pathlib, requests, shutil, zipfile

token = userdata.get('GITHUB_TOKEN')
if not token:
    raise RuntimeError('Ajoute le secret Colab GITHUB_TOKEN et autorise son accès au notebook.')

repo = 'massynas/kabyle-ai'
branch = 'feat/colab-first-run'
url = f'https://api.github.com/repos/{repo}/zipball/{branch}'
response = requests.get(url, headers={'Authorization': f'Bearer {token}', 'Accept': 'application/vnd.github+json'}, timeout=120)
response.raise_for_status()
archive = zipfile.ZipFile(io.BytesIO(response.content))
extract_root = pathlib.Path('/content/kabyle-ai-extract')
shutil.rmtree(extract_root, ignore_errors=True)
extract_root.mkdir(parents=True)
archive.extractall(extract_root)
repo_dirs = list(extract_root.iterdir())
if len(repo_dirs) != 1 or not repo_dirs[0].is_dir():
    raise RuntimeError('Archive GitHub inattendue')
project_dir = pathlib.Path('/content/kabyle-ai')
shutil.rmtree(project_dir, ignore_errors=True)
shutil.move(str(repo_dirs[0]), project_dir)
os.chdir(project_dir)
print('Code privé récupéré depuis la branche', branch)


## 1. Vérifier l'environnement Colab
Dans **Exécution → Modifier le type d'exécution**, sélectionne un GPU si Colab en propose un. Le test du tokenizer fonctionne aussi sans GPU.


In [ ]:
%cd /content/kabyle-ai
!python -m pip install -U pip
!python -m pip install -e '.[train]'


In [ ]:
import torch, platform
print('Python:', platform.python_version())
print('PyTorch:', torch.__version__)
print('CUDA disponible:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM (GiB):', round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))
else:
    print('Pas de GPU détecté : ne lance pas de fine-tuning QLoRA dans cette session.')


## 2. Vérifier le tokenizer Kabyle / français / arabe / tifinagh
Cette étape récupère uniquement les fichiers du tokenizer, pas les poids du modèle. La révision est résolue en SHA immuable et réutilisée ensuite.


In [ ]:
from huggingface_hub import HfApi
from transformers import AutoTokenizer

MODEL_ID = 'Qwen/Qwen3-1.7B'
MODEL_REVISION = HfApi().model_info(MODEL_ID).sha
print('Modèle :', MODEL_ID)
print('Révision immuable :', MODEL_REVISION)
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, revision=MODEL_REVISION, trust_remote_code=False)
samples = {
    'Kabyle latin': 'Axxam-nneɣ, ad nerr awal s teqbaylit. Ɛ, ḍ, ṛ, ṣ, ṭ, č, ɣ.',
    'Français': 'Bonjour, nous testons la traduction du kabyle vers le français.',
    'Arabe': 'مرحبا، نختبر الترجمة من القبائلية إلى العربية.',
    'Tifinagh': 'ⴰⵣⵓⵍ ⴰⵎⵙⵍⵉ ⵏ ⵜⵇⴱⴰⵢⵍⵉⵜ',
    'Mixte': 'Azul ! Bonjour ! مرحبا ! ⴰⵣⵓⵍ !'
}
for label, sample in samples.items():
    ids = tokenizer.encode(sample, add_special_tokens=False)
    decoded = tokenizer.decode(ids, skip_special_tokens=False)
    print(f'\n{label}: {len(ids)} tokens')
    print('Round-trip exact :', decoded == sample)
    if decoded != sample:
        print('Décodé :', repr(decoded))


## 3. Valider un petit fichier d'entraînement autorisé
Téléverse un JSONL préparé en amont. Chaque ligne doit respecter le schéma et porter une autorisation explicite d'entraînement avec une référence de preuve. Pour ce premier essai, commence avec un petit échantillon (par exemple 50 à 200 lignes) dont tu as vérifié les droits. Le fichier reste dans la session Colab et n'est pas ajouté à Git.


In [ ]:
from google.colab import files
uploaded = files.upload()
if not uploaded:
    print('Aucun fichier téléversé : étape d'entraînement ignorée.')
else:
    if len(uploaded) != 1:
        raise RuntimeError('Téléverse un seul fichier JSONL à la fois.')
    uploaded_name = next(iter(uploaded))
    training_file = pathlib.Path('/content') / uploaded_name
    if training_file.suffix.lower() != '.jsonl':
        raise RuntimeError('Le fichier doit être au format .jsonl')
    print('Fichier reçu :', training_file.name)


In [ ]:
import subprocess
if 'training_file' in globals():
    cmd = [
        'python', 'scripts/train_lora.py', '--input', str(training_file),
        '--model', MODEL_ID, '--revision', MODEL_REVISION,
        '--output-dir', '/content/kabyle-ai/outputs/colab-dry-run'
    ]
    subprocess.run(cmd, check=True)
else:
    print('Pas de fichier JSONL : validation ignorée.')


## 4. Fine-tuning QLoRA — désactivé par défaut
Ne passe à cette étape qu'après validation du JSONL, confirmation des droits et vérification que le GPU est disponible. Colab gratuit peut interrompre la session et manquer de VRAM. Les résultats restent temporaires tant qu'ils ne sont pas exportés.

Le premier essai est volontairement limité à une époque, une longueur de 512 tokens et un petit jeu de données. La cellule exige une activation manuelle de `RUN_TRAINING`.


In [ ]:
RUN_TRAINING = False  # Mets True uniquement après vérification des droits et du GPU.
if RUN_TRAINING is not True:
    print('Entraînement désactivé. Aucun poids téléchargé, aucune dépense engagée.')
elif 'training_file' not in globals():
    raise RuntimeError('Téléverse d’abord un JSONL approuvé.')
elif not torch.cuda.is_available():
    raise RuntimeError('Un GPU CUDA est obligatoire pour le test QLoRA.')
else:
    output_dir = '/content/kabyle-ai/outputs/qwen3-kabyle-colab-test'
    cmd = [
        'python', 'scripts/train_lora.py', '--input', str(training_file),
        '--model', MODEL_ID, '--revision', MODEL_REVISION,
        '--output-dir', output_dir, '--epochs', '1', '--batch-size', '1',
        '--gradient-accumulation-steps', '8', '--max-length', '512',
        '--load-in-4bit', '--confirm-training'
    ]
    subprocess.run(cmd, check=True)
    print('Entraînement terminé :', output_dir)


In [ ]:
if 'output_dir' in globals() and pathlib.Path(output_dir).exists():
    archive_path = shutil.make_archive('/content/kabyle-ai-colab-adapter', 'zip', output_dir)
    print('Archive créée :', archive_path)
    files.download(archive_path)
else:
    print('Aucun adaptateur à exporter.')
